# 05 — Read the report: provenance and unknowns

**Goal:** learn to read a comprehension report the way a reviewer reads a diff — starting from *how do you know?*

We use the bundled program-graph fixture (the shape Gecko produces for a Solana storefront program) because it exercises every provenance tier; the same reading skill applies to any comprehended API.

In [ ]:
# Live-mode guard: cells that invoke the real gecko CLI run only when you
# opt in AND npx is available. Everything else in this notebook is offline.
#   export GECKO_COOKBOOK_LIVE=1   # to enable live cells
import os
import shutil

GECKO_LIVE = os.environ.get("GECKO_COOKBOOK_LIVE") == "1" and shutil.which("npx") is not None
FIXTURES = None
from pathlib import Path
here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
FIXTURES = here / "cookbook" / "fixtures"
print(f"live mode: {GECKO_LIVE} — fixtures: {FIXTURES}")

## 1. The trust ladder

| Tier | Meaning | Audit question |
|---|---|---|
| **declared** | the IDL/spec says so | is the spec honest? |
| **recovered** | read from program source | does source match deployment? |
| **measured** | proven by execution | strongest — what moved? |
| **inferred** | a heuristic guess | labeled as a guess — verify before trusting |


In [ ]:
import json

graph = json.loads((FIXTURES / "program-graph-example.json").read_text())
print(f"program: {graph['program']}")
for instruction in graph['instructions']:
    print(f"\n{instruction['name']}: {instruction['summary']}")
    for account in instruction['accounts']:
        note = f"  <- {account['note']}" if 'note' in account else ''
        print(f"  {account['name']:22} {account['kind']:8} [{account['provenance']}]{note}")

## 2. Exercise the reviewer's questions

Look at the `purchase` instruction above and answer (cell below computes the ground truth):

1. Which account's derivation does the IDL **not** fully describe?
2. Which claim is backed by actual execution?
3. What must be read from chain **before** the buyer's token account can be derived?

In [ ]:
purchase = next(i for i in graph['instructions'] if i['name'] == 'purchase')

recovered = [a['name'] for a in purchase['accounts'] if a['provenance'] == 'recovered']
measured = [a['name'] for a in purchase['accounts'] if a['provenance'] == 'measured']
print(f"1. beyond the IDL (recovered from source): {recovered}")
print(f"2. proven by execution (measured): {measured} — "
      f"CU={purchase['measured']['compute_units']}, "
      f"movement={purchase['measured']['movement']}")
print("3. derivation order:")
for step in purchase['derivation_order']:
    print(f"   - {step}")

## 3. Why unknowns are a feature

A report that says *'item seeds recovered from source — the IDL omits item_index'* is telling you exactly where a naive integration would have failed: an agent working from the IDL alone **cannot construct that account**. Comprehension isn't magic; it's honest bookkeeping about what is known, how, and what remains a guess.

**Next:** `advanced/09` walks this same graph as a *routing* problem (find_start), and `advanced/10` executes the purchase on a fork and reads the receipt.